# TESDA AI/ML Accelerator - Introduction

Start here. This notebook orients the series: the problems it builds toward, the datasets, the
tools, and how each guided lab is structured.

### New to Python or pandas?

Anyone coming from another language can run `S2_commands_primer` first. It lays out the mental model in one page: a `DataFrame` is a table (a SQL table, an R `data.frame`, a spreadsheet), a `merge` is a `JOIN`, `NaN` is the missing value, and a chain of `.method().method()` reads left to right. The comments in the code cells point out each pandas idiom as it appears.

## What these notebooks build

The series works through three problems from the TESDA use cases, using real assessment-center,
program, assessor, boundary, and road-network data for the Philippines:

- **Center recommendation.** Given a learner (location and target qualification), rank the
  assessment centers that can serve them, by travel effort and capacity.
- **Gap analysis.** Find underserved areas: places where demand for a qualification is far from
  any accredited center.
- **Assessor scoping.** Measure assessor workload and coverage from activity records.

Each topic is a short, hands-on lab. The labs build on each other: clean and align the data first,
turn it into spatial features, then answer the problem.

## How each lab is structured

- Run every cell in order (`Shift+Enter`). Later cells depend on earlier ones.
- Exercise cells have `### START CODE HERE ###` and `### END CODE HERE ###`. Fill in the code
  between those markers.
- A check cell with `assert` statements follows each exercise and prints a pass or a fix hint.
- A question cell poses a concrete number to compute and report.
- Play-around cells hold a parameter to change and re-run.

Markers used in the labs: run ▶, exercise ✏️, check ✅, play-around 🎛️, map 🗺️, flaw to catch 🐞.

## The data

The datasets live in a public Google Cloud Storage bucket. Base URL:

`https://storage.googleapis.com/tesda-datasets/data/`

| File | Contents |
|------|----------|
| `2026 Accredited …Assessment Center… July 15, 2026 database.csv` | Accredited assessment centers (the raw registry) |
| `tesda_coordinates.parquet` | Geocoded institutions with PSGC codes and a coordinate-quality flag |
| `Compendium of Registered Programs as of June 2026.xlsx` | Registered programs and NC levels |
| `consolidated_geodata_matched.gpkg` | Barangay boundaries matched to PSGC (polygons) |
| `shapefiles/phl_admbnda_adm4_updated.shp` | Admin boundaries, barangay level (plus sibling files) |
| `ncr_clip.osm.pbf` | OpenStreetMap road extract for Metro Manila |
| `assessors.csv`, `assessor_qualifications.csv`, `assessor_timelogs.csv` | Assessor records and time logs |

## Set up the environment
In Colab, run the install cell once per session. Skip it locally if the packages are present.

In [ ]:
# Colab setup. Skip locally if already installed.
!pip -q install geopandas shapely pyproj rapidfuzz esda libpysal osmnx folium mapclassify pyarrow

## Confirm access to the data
Load the geocoded institutions table straight from the public bucket and inspect it.

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)   # show every column in previews
pd.set_option("display.width", None)         # do not wrap wide tables

BASE = "https://storage.googleapis.com/tesda-datasets/data/"
coords = pd.read_parquet(BASE + "tesda_coordinates.parquet")   # parquet stores column types, so numbers arrive as numbers
print("geocoded TESDA institutions:", coords.shape)
coords[["name", "region", "city_municipality", "latitude", "longitude", "coord_status"]].head()

**Expected output:** `geocoded TESDA institutions: (8007, 21)`, then a five-row preview with a
name, region, city or municipality, latitude, longitude, and a `coord_status` flag.

## A first look at each dataset

The catalog above names the files. Each real dataset below reports its **shape**, its **columns and types**, a **numeric summary** (`describe`), and the **first rows**, so the schema is clear before any lab depends on it. A small `preview` helper keeps the report uniform.

Personal-data columns (names, phone numbers, email) are dropped on load and never carried further.

In [ ]:
import os
from urllib.parse import quote
import geopandas as gpd

def source(name):                        # percent-encode spaces in a filename for an HTTP read
    return BASE + quote(name)

VSI = "/vsicurl/" + BASE                 # GDAL reads the large vector files in place over HTTP
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".gpkg,.shp,.shx,.dbf,.prj,.cpg"
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"

def preview(df, name):
    """Report shape, columns with dtypes, and a summary of both numeric and text columns."""
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print("columns:", list(df.columns))
    display(df.dtypes.to_frame("dtype").T)        # one dtype per column, laid out as a single row
    num = df.select_dtypes("number")              # keep only the numeric columns
    if num.shape[1]:                              # .shape[1] is the column count; skip if none
        print("numeric columns (count, mean, spread, range):")
        display(num.describe().T.round(3))        # describe() = count/mean/std/quartiles per column; .T puts columns as rows
    text = df.select_dtypes(exclude="number").drop(columns="geometry", errors="ignore")   # errors="ignore": fine if no geometry column
    if text.shape[1]:
        print("text columns (count, distinct values, most frequent):")
        display(text.astype("object").describe().T)   # describe() on text gives count/unique/top/freq instead
    return df.head()

### Geocoded institutions - `tesda_coordinates.parquet`

The backbone table, already loaded above as `coords`. Parquet stores column types, so `latitude` and `longitude` arrive as numbers and `describe` can summarize them.

In [ ]:
preview(coords, "tesda_coordinates.parquet")

**Expected output:** 8,007 rows across 21 columns, then the dtypes and the numeric summary. Read the `latitude`/`longitude` summary closely: a valid latitude falls in [-90, 90] and a Philippine longitude near [116, 127], yet the `max` runs to the billions. Those impossible values are exactly what the coordinate-quality audit in `S2_D1_03` detects and repairs. The `coord_status` column already flags each row.

### Accredited assessment centers - the raw registry (CSV)

The government accreditation export. One row is one center-qualification accreditation, so a center appears on many rows. A CSV stores no types, so every column arrives as text (note the misspelled `Longtitude`/`Latitude` headers, kept as the source has them).

In [ ]:
ac = pd.read_csv(
    source("2026 Accredited Compentency Assessment Center based on July 15, 2026 database.csv"),
    dtype=str, encoding="utf-8-sig",             # utf-8-sig drops the leading BOM some exports add
).drop(columns=["Center Manager", "Phone"])     # personal data, never kept
preview(ac, "Accredited Assessment Centers (CSV)")

**Expected output:** 12,871 rows across 14 columns, all of dtype text. The numeric summary is skipped (a CSV carries no types until the values are parsed), so the text summary reports each column's count, number of distinct values, and most frequent value instead.

### Registered programs - `Compendium of Registered Programs as of June 2026.xlsx`

The registered-program compendium, read from its data sheet. One row is one registered program at an institution.

In [ ]:
comp = pd.read_excel(
    source("Compendium of Registered Programs as of June 2026.xlsx"),
    sheet_name="Compendium ao June 2026", dtype=str,
).drop(columns=["TEL. NO.", "E-mail Address", "Name of Trainer/s", "NTTC No."])
preview(comp, "Registered Programs (Excel)")

**Expected output:** 21,967 rows across 19 columns, all text, so the summary reports count, distinct values, and the most frequent value for each column. The workbook may warn about a few out-of-range date cells. The read still completes.

### Assessor records - three linked CSVs (sample data)

Assessor scoping draws on three tables that link on `assessor_id`: the roster, the qualifications each assessor is certified for, and a time log of assessment sessions. These records are a generated sample.

In [ ]:
assessors = pd.read_csv(source("assessors.csv")).drop(columns=["full_name", "mobile_no"])
quals     = pd.read_csv(source("assessor_qualifications.csv"))
logs      = pd.read_csv(source("assessor_timelogs.csv"))
print("assessors:", assessors.shape, "| qualifications:", quals.shape, "| time logs:", logs.shape)
preview(logs, "Assessor time logs (CSV)")

**Expected output:** `assessors: (526, 8) | qualifications: (983, 5) | time logs: (47895, 8)`, then the time-log summary. `candidates_assessed` is numeric, so `describe` reports it. Its count is below the row total because non-assessment activities log no candidate count.

### Administrative boundaries - GeoPackage and shapefile

Barangay boundaries arrive as vector layers: a matched GeoPackage with PSGC codes, and the raw PSA/HDX shapefile. Both are hundreds of megabytes, so `/vsicurl/` opens them in place and `rows=` reads only the first features, enough to see the schema. The geometry column holds polygons, which have no numeric summary, so inspect the columns, the CRS, and the geometry type instead.

In [ ]:
brgy_gpkg = gpd.read_file(VSI + "consolidated_geodata_matched.gpkg", rows=50)
brgy_shp  = gpd.read_file(VSI + "shapefiles/phl_admbnda_adm4_updated.shp", rows=50)

# .crs is the coordinate reference system; .geom_type reports each row's geometry kind (point/line/polygon)
print("GeoPackage:", brgy_gpkg.shape[1], "columns | CRS", brgy_gpkg.crs, "|", brgy_gpkg.geom_type.iloc[0])
print("columns:", list(brgy_gpkg.columns))
print()
print("Shapefile: ", brgy_shp.shape[1], "columns | CRS", brgy_shp.crs, "|", brgy_shp.geom_type.iloc[0])
print("columns:", list(brgy_shp.columns))
brgy_gpkg[["psgc_code", "name", "adm1_en", "adm2_en", "adm3_en", "geometry"]].head()

**Expected output:** the GeoPackage reports 23 columns and the shapefile 19, both in EPSG:4326, with MultiPolygon and Polygon geometry, then a few named barangay polygons. The shapefile shortens field names to 10 characters (`Shape_Leng`), one of its known limits.

## The tools

- **pandas, geopandas, shapely, pyproj**: tabular and geospatial data, geometry, coordinate
  reference systems.
- **osmnx, networkx, OSRM**: road networks and travel times for routing.
- **folium, matplotlib, mapclassify**: interactive and static maps.
- **esda, libpysal**: spatial statistics such as Moran's I.
- **rapidfuzz**: name matching when records share no common identifier.

## Run the notebooks

- **Colab:** open a notebook through its badge in the README. The first cell installs what it
  needs, and the data loads from the public bucket.
- **Local:** `pip install -r requirements.txt`, then open a notebook in Jupyter or VS Code.

Read the labs in order. The first labs clean and align the center data, the middle labs turn it
into spatial features, and the later labs answer the three problems.

## References & further reading

Documentation for the main tools.

- [GeoPandas, user guide & API](https://geopandas.org/)
- [Shapely (geometry)](https://shapely.readthedocs.io/)
- [pyproj / CRS](https://pyproj4.github.io/pyproj/stable/)
- [pandas, user guide](https://pandas.pydata.org/docs/user_guide/)
- [OSMnx, street networks from OpenStreetMap](https://osmnx.readthedocs.io/)
- [OSRM, routing engine](https://project-osrm.org/)
- [Folium, interactive maps](https://python-visualization.github.io/folium/)
- [PySAL / esda (Moran's I)](https://pysal.org/esda/)
- [PSA, Philippine Standard Geographic Code (PSGC)](https://psa.gov.ph/classification/psgc)